# `PSF_zlen` convergence of image formation

Goal: determine what `imghp.PSF_zlen` (number of z-nodes, evenly spaced across
`PSF_zlb_μm`/`PSF_zub_μm`, at which exact PSFs are computed -- any object pixel's true depth
between two nodes is interpolated) is needed for the forward-modeled image to converge.

**The heavy computation lives in `scripts/psf_zlen_convergence.jl`, run via
`sbatch cluster_scripts/run_psf_zlen_convergence.sh`, not here** -- same reasoning as the
`freq_order` convergence study (`2026-11-05-test-frequency-order-convergence.ipynb`): this notebook
just loads that script's saved `results.jld2` and plots, so it's safe to run in a small interactive
session.

Recap of what the script holds fixed, to isolate only the effect of `PSF_zlen`: `freq_order = 40`
throughout (the value recommended by the `freq_order` convergence study, so frequency-quadrature
error doesn't leak into this comparison); `PSF_zlb_μm`/`PSF_zub_μm` fixed at the production range
(`-2.1e6`/`-0.9e6` -- only the node count within it varies); a `monochromatic` lens and a `random`
lens (each a single fixed `geoms`, reused unchanged across every `PSF_zlen`); noiseless images
only; and a `PSF_scale` chosen once per geometry (at `PSF_zlen=3`) and held fixed across the sweep.

Unlike the `freq_order` study -- which deliberately used a single fixed object depth to avoid
confounding -- this one deliberately uses a `UniformlyRandomObject` (random `Tmap` *and* `zmap`,
spanning `-2.0e6` to `-1.0e6`, with its own `zlen=1000` so its z-grid is effectively continuous),
so a single formed image exercises interpolation quality across many different z-values at once.
`PSF_zlen` is unrelated to any object_type's own z-sampling density -- see the script's header
comment for why.

**Note:** an earlier run of this study (before commit `c9c7792`) was confounded by a real bug --
`get_discretized_δ_function`'s z-interpolation kernel was normalized as a discretized Dirac delta
(carrying an extra `1/Δz` factor) rather than a partition of unity, so `make_image_from_3D` /
`make_image_from_3D_distributed` silently scaled the formed image by roughly the number of
`PSF_zlen` nodes, with no corresponding convergence. That's fixed now (see `src/forward.jl`'s
`get_discretized_δ_function` and `src/backward.jl`'s `dδ_dz`/`d2δ_dz2`), so the plain relative-L2
comparisons below are directly meaningful -- no manual rescaling needed.

## Load precomputed results

In [ ]:
cd("../../End2EndThermalImg.jl")
import Pkg
Pkg.activate(".")

In [ ]:
using JLD2
using LinearAlgebra
using Statistics
using PythonPlot

Picks the most recent `convergence_studies/psf_zlen_<datetime>/` run folder by default --
set `run_folder` explicitly instead if you want a specific earlier run.

In [ ]:
root = "convergence_studies"
run_names = sort(filter(name -> startswith(name, "psf_zlen_"), readdir(root)))
run_folder = joinpath(root, run_names[end])
println("Loading results from: ", run_folder)

results_file = joinpath(run_folder, "results.jld2")
psf_zlens, geoms_mono, geoms_random, object, PSF_scale_mono, PSF_scale_random, images, php, imghp_ref =
    JLD2.load(results_file, "psf_zlens", "geoms_mono", "geoms_random", "object", "PSF_scale_mono",
              "PSF_scale_random", "images", "php", "imghp_ref")

labels = ["monochromatic", "random"]
println("psf_zlens = ", psf_zlens)
println("PSF_scale_mono = ", PSF_scale_mono, "  PSF_scale_random = ", PSF_scale_random)

## Convergence error

Relative L2 error of each `PSF_zlen`'s image against the finest one computed (`PSF_zlen=33`,
treated as the reference/"converged" image), separately for each geometry.

In [ ]:
reference_zlen = maximum(psf_zlens)
psf_zlens_sorted = sort(psf_zlens)

rel_errors = Dict{String, Vector{Float64}}()
for label in labels
    image_ref = images[(label, reference_zlen)]
    rel_errors[label] = [norm(images[(label, zlen)] .- image_ref) / norm(image_ref) for zlen in psf_zlens_sorted]
end

for label in labels
    println(label, ":")
    for (zlen, err) in zip(psf_zlens_sorted, rel_errors[label])
        println("  PSF_zlen = $zlen  relative L2 error (vs PSF_zlen=$reference_zlen) = $err")
    end
end

In [ ]:
fig, ax = subplots(figsize=(6, 5))
zlens_to_plot = filter(zlen -> zlen != reference_zlen, psf_zlens_sorted)
for label in labels
    errs_to_plot = [rel_errors[label][findfirst(==(zlen), psf_zlens_sorted)] for zlen in zlens_to_plot]
    ax.loglog(zlens_to_plot, errs_to_plot, marker="o", label=label)
end
ax.set_xticks(zlens_to_plot)
ax.set_xticklabels(string.(zlens_to_plot))
ax.set_xlabel("PSF_zlen")
ax.set_ylabel("relative L2 error (vs PSF_zlen = $reference_zlen)")
ax.set_title("PSF_zlen convergence of image formation")
ax.legend()
ax.grid(true, which="both", alpha=0.3)
fig;

## Error between consecutive `PSF_zlen` values

Relative L2 error between each `PSF_zlen` and the next one up in the sweep, separately for each
geometry. Decreasing successive error is itself evidence of convergence, independent of the choice
of `PSF_zlen=33` as a reference.

In [ ]:
zlen_pairs = collect(zip(psf_zlens_sorted[1:end-1], psf_zlens_sorted[2:end]))

rel_errors_consecutive = Dict{String, Vector{Float64}}()
for label in labels
    rel_errors_consecutive[label] = [norm(images[(label, zlen2)] .- images[(label, zlen1)]) / norm(images[(label, zlen2)]) for (zlen1, zlen2) in zlen_pairs]
end

for label in labels
    println(label, ":")
    for ((zlen1, zlen2), err) in zip(zlen_pairs, rel_errors_consecutive[label])
        println("  PSF_zlen $zlen1 -> $zlen2  relative L2 error = $err")
    end
end

In [ ]:
fig, ax = subplots(figsize=(6, 5))
xs = 1:length(zlen_pairs)
for label in labels
    ax.semilogy(xs, rel_errors_consecutive[label], marker="o", label=label)
end
ax.set_xticks(xs)
ax.set_xticklabels(["$(zlen1)→$(zlen2)" for (zlen1, zlen2) in zlen_pairs])
ax.set_xlabel("PSF_zlen transition")
ax.set_ylabel("relative L2 error between consecutive PSF_zlen")
ax.set_title("Image change between consecutive PSF_zlen values")
ax.legend()
ax.grid(true, which="both", alpha=0.3)
fig;

## Visual comparison

Image at every `PSF_zlen` in the sweep, separately for each geometry.

In [ ]:
for label in labels
    fig, axes = subplots(1, length(psf_zlens_sorted); figsize=(4 * length(psf_zlens_sorted), 4))
    for (col, zlen) in enumerate(psf_zlens_sorted)
        im = axes[col - 1].imshow(images[(label, zlen)])
        axes[col - 1].set_title("PSF_zlen=$zlen")
        fig.colorbar(im, ax=axes[col - 1])
    end
    fig.suptitle(label)
    fig.tight_layout()
    display(fig)
end